# Triagem medidor × outros sem rótulos (rubrica de ML)

Notebook dos experimentos do artigo. O protocolo está em `ml_README.md`,
seção 1, e foi fixado antes de qualquer experimento.

**Como executar.** As fotos do cliente não fazem parte do repositório. O
notebook lê a pasta de dados e a pasta de trabalho indicadas no `.env` da raiz
deste repositório, e só reexecuta na máquina que tem esses dados. As saídas
mostram apenas contagens, métricas e gráficos em SVG.

## 1. Configuração

In [1]:
# Configuração: versões e impressões digitais dos artefatos de entrada.
# Nenhum caminho é exibido (decisão E15): só versões e SHA-256.
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch

from meter_reader.config import get_data_dir, get_work_dir
from training.build_manifest import MANIFEST_NAME, file_sha256
from training.build_sample import SAMPLE_NAME
from training.labels import load_labeled_photos

# Rótulos do Lab02, os mesmos da rubrica de ML (E11); o notebook roda de ml-artigo/
LABELS_PATH = Path("../lab-02/rotulos/lab2_rotulos.csv")
assert LABELS_PATH.is_file(), "Execute o notebook a partir da pasta ml-artigo/."

WORK_DIR = get_work_dir()   # valida o .env; não exibe o caminho
DATA_DIR = get_data_dir()

print(f"Python       {sys.version.split()[0]}")
print(f"PyTorch      {torch.__version__} | timm {timm.__version__}")
print(f"GPU          {torch.cuda.is_available()}")
print(f"SHA-256 manifesto: {file_sha256(WORK_DIR / MANIFEST_NAME)}")
print(f"SHA-256 amostra:   {file_sha256(WORK_DIR / SAMPLE_NAME)}")
print(f"SHA-256 rótulos:   {file_sha256(LABELS_PATH)}")

# Fotos rotuladas fora do teste (o teste segue lacrado: include_test=False)
photos = load_labeled_photos(LABELS_PATH)
random_sample = [p for p in photos if p.role != "estresse_zero"]
print(f"Rotuladas fora do teste: {len(photos)} | amostra aleatória: {len(random_sample)}")

c:\Projetos_Tec\leitura_automatizada_mostradores_energia_pesquisa\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python       3.11.4
PyTorch      2.14.1+cu126 | timm 1.0.30
GPU          True
SHA-256 manifesto: 56467c02e64f41b6c495987e4cec4c0216dc1e38e5644ae65bceca8d2199b28a
SHA-256 amostra:   a6ced4666242f3c5832d5109c1fdc9289bbc754a693a9252971ed5259c417796
SHA-256 rótulos:   b901c05c2b5e6b89fba53adf512681a8d2507ef1aa9c2597033b6f84069c2f2e
Rotuladas fora do teste: 261 | amostra aleatória: 225


## 2. Extrator: custo e paridade

Custo por foto em CPU (4 threads, piso da E04), só o modelo, sobre 50 fotos da
amostra aleatória: DINOv2 ViT-S/14 (proposta) contra a MobileNetV3-Large, a
espinha da triagem de DL. Paridade: diferença entre as características
calculadas na GPU (onde o cache é extraído) e na CPU, sem TF32 (E33).

In [2]:
# Custo em CPU: parâmetros, operações e latência por foto (só o modelo)
import logging
import time

from PIL import Image
from torch.utils.flop_counter import FlopCounterMode

from meter_reader.preprocess import enhance, preprocess
from meter_reader.triage import TriageModel
from training.dino_features import extract_views, load_dinov2, model_input

logging.getLogger("torch.utils.flop_counter").setLevel(logging.ERROR)
THREADS = 4  # piso da E04
timing_photos = random_sample[:50]


def latency_ms(fn, items, warmup=10, runs=200):
    """p50 e p95 em milissegundos, alternando entre os itens."""
    for k in range(warmup):
        fn(items[k % len(items)])
    times = []
    for k in range(runs):
        start = time.perf_counter()
        fn(items[k % len(items)])
        times.append((time.perf_counter() - start) * 1000)
    return np.percentile(times, [50, 95])


def dino_tensor(path):
    with Image.open(path) as image:
        return model_input(enhance(image))


dino = load_dinov2()  # primeira vez: baixa os pesos oficiais
mobile = TriageModel("mobilenet_v3_large")
mobile.freeze_backbone()
mobile.eval()

extractors = {
    "DINOv2 ViT-S/14": (dino, dino_tensor, dino.parameters()),
    "MobileNetV3-Large": (mobile.embed, lambda p: preprocess(p, 224), mobile.features.parameters()),
}

previous_threads = torch.get_num_threads()
torch.set_num_threads(THREADS)
rows = {}
try:
    with torch.no_grad():
        for name, (fn, prepare, params) in extractors.items():
            tensors = [prepare(p.path).unsqueeze(0) for p in timing_photos]
            with FlopCounterMode(display=False) as counter:
                dim = fn(tensors[0]).shape[1]
            p50, p95 = latency_ms(fn, tensors)
            rows[name] = {
                "entrada (A × L)": f"{tensors[0].shape[2]} × {tensors[0].shape[3]}",
                "dimensão": dim,
                "parâmetros (M)": round(sum(t.numel() for t in params) / 1e6, 2),
                "GMAC": round(counter.get_total_flops() / 2e9, 2),  # convenção do torchvision
                "p50 (ms)": round(p50, 1),
                "p95 (ms)": round(p95, 1),
            }
finally:
    torch.set_num_threads(previous_threads)

cost = pd.DataFrame(rows).T
cost

W1020 21:31:35.921000 38668 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels
c:\Projetos_Tec\leitura_automatizada_mostradores_energia_pesquisa\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pheli\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for

,entrada (A × L),dimensão,parâmetros (M),GMAC,p50 (ms),p95 (ms)
DINOv2 ViT-S/14,294 × 224,384,22.06,7.23,95.6,150.4
MobileNetV3-Large,299 × 224,960,2.97,0.29,19.2,30.5


In [3]:
# Paridade GPU x CPU das características DINOv2, sem TF32 (E33)
assert torch.cuda.is_available(), "A paridade exige a GPU."
parity_photos = random_sample[:16]
paths = [p.path for p in parity_photos]
codes = [p.pseudonym for p in parity_photos]

cpu_features = extract_views(paths, codes, 0, dino, "cpu")
gpu_features = extract_views(paths, codes, 0, dino, "cuda")
dino.to("cpu")  # extract_views move o modelo; volta para a CPU

difference = np.abs(gpu_features - cpu_features)
relative = difference.max() / np.abs(cpu_features).max()
cosine = (gpu_features * cpu_features).sum(-1) / (
    np.linalg.norm(gpu_features, axis=-1) * np.linalg.norm(cpu_features, axis=-1))
print(f"Fotos: {len(parity_photos)}")
print(f"Diferença máxima absoluta: {difference.max():.2e} | relativa: {relative:.2e}")
print(f"Similaridade de cosseno mínima GPU x CPU: {cosine.min():.8f}")

Fotos: 16
Diferença máxima absoluta: 3.53e-05 | relativa: 3.56e-06
Similaridade de cosseno mínima GPU x CPU: 0.99999988


## 3. Características do universo do agrupamento

Todas as fotos íntegras dos três lotes de desenvolvimento, órfãs incluídas
(protocolo, seção 1.1): a foto original e 5 vistas aumentadas (seção 1.3),
extraídas uma vez na GPU, sem TF32, e guardadas na pasta de trabalho. O lote
de teste não é lido aqui.

In [4]:
# Cache das características DINOv2: foto original + 5 vistas, lotes de desenvolvimento
from training.dino_features import cache_path, load_or_extract_views
from training.unlabeled import load_development_photos

N_VIEWS = 5
CACHE_NAME = "dinov2_desenvolvimento"
cache_dir = WORK_DIR / "caracteristicas"

dev_photos = load_development_photos()
dev_codes = [p.pseudonym for p in dev_photos]

start = time.perf_counter()
dev_features = load_or_extract_views([p.path for p in dev_photos], dev_codes, CACHE_NAME,
                                     N_VIEWS, cache_dir, device="cuda")
dino.to("cpu")
elapsed_min = (time.perf_counter() - start) / 60

# Conferências
position = {code: i for i, code in enumerate(dev_codes)}
labeled_codes = {p.pseudonym for p in photos}
assert labeled_codes <= position.keys(), "Há fotos rotuladas fora do universo."
assert np.isfinite(dev_features).all(), "Características com NaN ou infinito."
parity_rows = dev_features[[position[c] for c in codes], 0]   # mesma GPU, mesmas fotos
drift = np.abs(parity_rows - gpu_features[:, 0]).max()

print(f"Fotos por lote: {pd.Series([p.lot for p in dev_photos]).value_counts().sort_index().to_dict()}")
print(f"Total: {len(dev_photos)} | forma do cache: {dev_features.shape}")
print(f"Rotuladas no universo: {len(labeled_codes)} de {len(photos)}")
print(f"Diferença para a paridade (foto original): {drift:.2e}")
print(f"Tempo: {elapsed_min:.1f} min")
print(f"SHA-256 do cache: {file_sha256(cache_path(cache_dir, CACHE_NAME, N_VIEWS))}")

Fotos por lote: {'2026-05-20': 3181, '2026-05-21': 3094, '2026-05-22': 3050}
Total: 9325 | forma do cache: (9325, 6, 384)
Rotuladas no universo: 261 de 261
Diferença para a paridade (foto original): 1.96e-05
Tempo: 14.6 min
SHA-256 do cache: 17e3777351c296597ee8b29dee0cfd9ce36177ff32affa4c2fffada4ab51ef94


## 4. Pureza dos vizinhos por extrator (H2)

Pureza dos 5 vizinhos (cosseno) de cada foto da amostra aleatória, buscados
entre as próprias 225 fotos, no colapso binário medidor × outros. Estatística
de decisão: pureza balanceada (protocolo, seção 1.7). DINOv2 vence um extrator
se o IC 95% da diferença, por bootstrap pareado e estratificado por classe,
excluir 0. A baseline é padronizada; as redes, não (seção 1.4).

In [5]:
# H2: pureza dos vizinhos, DINOv2 x MobileNetV3-Large x baseline manual
from training.evaluation import OUTROS
from training.features import load_or_extract
from training.neighbors import (balanced_purity, cosine_neighbors, neighbor_purity,
                                paired_bootstrap_purity, standardize)

K_PURITY = 5
sample_idx = np.array([i for i, p in enumerate(photos) if p.role != "estresse_zero"])
is_outros = np.array([p.scene == OUTROS for p in photos])[sample_idx].astype(int)
assert len(sample_idx) == 225 and is_outros.sum() == 17

# Mesmas 225 fotos, na mesma ordem, nos três extratores
spaces = {
    "DINOv2 ViT-S/14": dev_features[[position[photos[i].pseudonym] for i in sample_idx], 0],
    "MobileNetV3-Large": load_or_extract(photos, "mobilenet_v3_large_224", cache_dir)[sample_idx],
    "Baseline manual": standardize(load_or_extract(photos, "baseline_224", cache_dir)[sample_idx]),
}
purity = {name: neighbor_purity(cosine_neighbors(x, K_PURITY), is_outros)
          for name, x in spaces.items()}

# Nível do acaso: vizinhos sorteados entre as outras 224 fotos
n = len(is_outros)
chance_meter = (n - is_outros.sum() - 1) / (n - 1)
chance_outros = (is_outros.sum() - 1) / (n - 1)

rows = {
    name: {
        "dimensão": spaces[name].shape[1],
        "pureza medidor": round(p[is_outros == 0].mean(), 3),
        "pureza outros": round(p[is_outros == 1].mean(), 3),
        "pureza balanceada": round(balanced_purity(p, is_outros), 3),
    }
    for name, p in purity.items()
}
rows["Acaso (referência)"] = {
    "dimensão": "-", "pureza medidor": round(chance_meter, 3),
    "pureza outros": round(chance_outros, 3),
    "pureza balanceada": round((chance_meter + chance_outros) / 2, 3),
}
print(f"Fotos: {n} (medidor {n - is_outros.sum()}, outros {is_outros.sum()}) | k = {K_PURITY}\n")
print(pd.DataFrame(rows).T.to_string())

# Decisão da H2: DINOv2 contra cada um dos outros extratores
print("\nDiferença da pureza balanceada (DINOv2 - outro), IC 95%:")
for other in ("MobileNetV3-Large", "Baseline manual"):
    point, low, high = paired_bootstrap_purity(purity["DINOv2 ViT-S/14"], purity[other], is_outros)
    verdict = "DINOv2 vence" if low > 0 else ("DINOv2 perde" if high < 0 else "empate")
    print(f"  x {other:18} {point:+.3f}  [{low:+.3f}; {high:+.3f}]  -> {verdict}")

Fotos: 225 (medidor 208, outros 17) | k = 5

                   dimensão pureza medidor pureza outros pureza balanceada
DINOv2 ViT-S/14       384.0          0.979         0.506             0.742
MobileNetV3-Large     960.0          0.979           0.6             0.789
Baseline manual      1859.0          0.929         0.271               0.6
Acaso (referência)        -          0.924         0.071             0.498

Diferença da pureza balanceada (DINOv2 - outro), IC 95%:
  x MobileNetV3-Large  -0.047  [-0.088; -0.006]  -> DINOv2 perde
  x Baseline manual    +0.143  [+0.050; +0.228]  -> DINOv2 vence
